### E-Commerce US Dataset

\Importing Necessary Libraries

In [9]:
import numpy as np
import pandas as pd
import os
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

\ Configuration - paths

In [10]:
RAW_PATH = r"c:\NG\E-Commerce US dataset\E-Commerce-US-dataset\data\raw\\"
 
SAMPLE_SIZE = 10000 

FILES = [
    "customers.csv", "orders.csv", "order_items.csv",
    "order_payments.csv", "order_reviews.csv",
    "products.csv", "sellers.csv", "geolocation.csv"
]

print(f"Config set! Sample size = {SAMPLE_SIZE} rows per logical group")

Config set! Sample size = 10000 rows per logical group


\Smart Relational Sampling

In [11]:
orders = pd.read_csv(RAW_PATH + "orders.csv", nrows=SAMPLE_SIZE)

order_ids = set(orders["order_id"])
customer_ids = set(orders["customer_id"])

print(f"Loaded orders: {orders.shape[0]} rows")

def load_filtered(filename, key_column, valid_ids):
    df = pd.read_csv(RAW_PATH + filename)
    filtered = df[df[key_column].isin(valid_ids)].reset_index(drop=True)
    return filtered

# order_id 
order_items    = load_filtered("order_items.csv",    "order_id", order_ids)
order_payments = load_filtered("order_payments.csv", "order_id", order_ids)
order_reviews  = load_filtered("order_reviews.csv",  "order_id", order_ids)

# customer_id 
customers = load_filtered("customers.csv", "customer_id", customer_ids)

# order_items
product_ids = set(order_items["product_id"])
seller_ids  = set(order_items["seller_id"])

products = load_filtered("products.csv", "product_id", product_ids)
sellers  = load_filtered("sellers.csv",  "seller_id",  seller_ids)

# zip codes 
zips = set(customers["customer_zip_code_prefix"]) | set(sellers["seller_zip_code_prefix"])
geo = pd.read_csv(RAW_PATH + "geolocation.csv")
geolocation = geo[geo["zip_code_prefix"].isin(zips)].reset_index(drop=True)

print("Sampling complete! All tables linked.")

Loaded orders: 10000 rows
Sampling complete! All tables linked.


In [12]:
tables = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "products": products,
    "sellers": sellers,
    "geolocation": geolocation,
}

print("===== SAMPLED DATA SUMMARY =====")
for name, df in tables.items():
    print(f"{name:18s} : {df.shape[0]:>6,} rows x {df.shape[1]} cols")

===== SAMPLED DATA SUMMARY =====
customers          : 10,000 rows x 9 cols
orders             : 10,000 rows x 8 cols
order_items        : 21,838 rows x 8 cols
order_payments     : 11,474 rows x 5 cols
order_reviews      :  9,332 rows x 7 cols
products           :  1,984 rows x 10 cols
sellers            :    500 rows x 8 cols
geolocation        : 11,255 rows x 5 cols


\Data Inspection

In [13]:
def explore_table(name, df):
    print("=" * 60)
    print(f"TABLE: {name}  |  Shape: {df.shape}")
    print("=" * 60)
    print("\n--- First 3 rows ---")
    print(df.head(3))
    print("\n--- Column dtypes ---")
    print(df.dtypes)
    print("\n")


In [14]:
explore_table("orders", orders)

TABLE: orders  |  Shape: (10000, 8)

--- First 3 rows ---
                               order_id                           customer_id order_status order_purchase_timestamp    order_approved_at order_delivered_carrier_date order_delivered_customer_date  \
0  720fd9e9-cdeb-4dec-a187-f71586eb085a  1e2e2881-a0eb-4cb0-829f-a566e810d05f     canceled      2025-12-27 07:07:20  2025-12-27 08:33:20                          NaN                           NaN   
1  c0142972-63fa-4af2-8070-f583ab769847  380b7418-308c-4bf7-b2bd-3ee446cb9ea6    delivered      2019-06-07 19:30:44  2019-06-08 05:08:44          2019-06-09 05:08:44           2019-06-14 05:08:44   
2  11bdf634-2b87-4d37-8d76-be1e7aff8f3b  89b0d980-868f-478d-a63b-5fea5f265f4f    delivered      2023-04-02 14:39:33  2023-04-02 23:42:33          2023-04-03 23:42:33           2023-04-08 23:42:33   

  order_estimated_delivery_date  
0           2026-01-04 08:33:20  
1           2019-06-16 05:08:44  
2           2023-04-10 23:42:33  

--- Colu